# Titanic — Binary Classification
**Week 5 | Phase 2 — ML Foundations | DS/ML Roadmap**

**Business question:** Can we predict which passengers survived the Titanic disaster, using demographic and ticketing data? A model that answers this demonstrates the full binary classification workflow: EDA, preprocessing, model selection, and evaluation beyond accuracy.

**Models compared:** Logistic Regression · Random Forest · K-Nearest Neighbours

**Evaluation:** Confusion matrix · Precision · Recall · F1 · ROC-AUC

## Section 1 — Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
)

# --- constants ---
RANDOM_STATE = 42
TEST_SIZE    = 0.2
DATA_PATH    = '../data/train.csv'

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"Using random state: {RANDOM_STATE}")
print(f"Test size: {TEST_SIZE}")
print(f"Data path: {DATA_PATH}")

Using random state: 42
Test size: 0.2
Data path: ../data/train.csv


## Section 2 — Data Loading & First Look

In [2]:
# Load the dataset
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

Loaded 891 rows, 12 columns


In [3]:
# Describing the dataset
print("Shape:", df.shape)
print("\nInfo:")
df.info()
print("\nDescriptive Stats:")
display(df.describe())
missing = df.isnull().sum()
print("\nMissing Values:")
display(missing[missing > 0].sort_values(ascending=False))

Shape: (891, 12)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Descriptive Stats:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200



Missing Values:


Cabin       687
Age         177
Embarked      2
dtype: int64

In [4]:
# Class balance
counts = df['Survived'].value_counts()
rate = df['Survived'].mean() * 100
print(f"Survived:         {counts[1]} ({rate:.1f}%)")
print(f"Did not survive:  {counts[0]} ({100 - rate:.1f}%)")

Survived:         342 (38.4%)
Did not survive:  549 (61.6%)


### Observations:
- 891 rows, 12 columns. Survived is the target, binary, no nulls.
- Dataset is imbalanced: 62% did not survive. A model that always predicts "dead" scores 61.6% accuracy without learning anything. This is why we use precision, recall, and AUC.
- Three missing value problems at very different scales. Cabin is 77% missing so we drop it. Age is 20% missing so we impute with the median. Embarked has just 2 missing rows so we impute with the mode.
- Fare is right-skewed. Median is £14, mean is £32, max is £512. A handful of very expensive tickets pull the average up significantly.
- Most passengers travelled alone. Median SibSp and Parch are both 0.

## Section 3 — Exploratory Data Analysis

In [5]:
# Survival rate by Sex
# your code here

*Observation:*
<!-- Fill in after running -->

In [6]:
# Survival rate by Pclass
# your code here

*Observation:*
<!-- Fill in after running -->

In [7]:
# Age distribution by survival
# your code here

*Observation:*
<!-- Fill in after running -->

In [8]:
# Fare distribution by Pclass
# your code here

*Observation:*
<!-- Fill in after running -->

In [9]:
# Missing values heatmap
# your code here

*Observation:*
<!-- Fill in after running -->

## Section 4 — Preprocessing

**Decisions documented here:**

| Column | Action | Reason |
|---|---|---|
| PassengerId | Drop | Row identifier, no signal |
| Name | Drop | Free text, not encoded this session |
| Ticket | Drop | High-cardinality, no clear signal |
| Cabin | Drop | >75% missing — imputation would be noise |
| Age | Impute (median) | ~20% missing; median robust to skew |
| Embarked | Impute (mode) | 2 rows missing; mode is safe |
| Sex | Binary encode (0/1) | Nominal, two categories |
| Embarked | One-hot encode | Nominal, three categories |

**Scaling strategy:** Logistic Regression and KNN receive scaled features (StandardScaler fit on train only). Random Forest receives the same encoded features unscaled — tree splits are scale-invariant, so scaling adds nothing and would obscure the deliberate comparison.

In [10]:
# Preprocessing: drop, impute, encode
# your code here
# should produce X and y

In [11]:
# Stratified train/test split + scale
# your code here
# should produce X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test

## Section 5 — Baseline (DummyClassifier)

Before any real model, establish a floor. A DummyClassifier predicting the majority class every time sets the minimum bar — any model that can't beat this is useless.

In [12]:
# DummyClassifier baseline
# your code here

## Section 6 — Model 1: Logistic Regression

Logistic Regression is the natural first real model for binary classification. It's interpretable (coefficients map to log-odds), trains fast, and gives well-calibrated probabilities. Requires scaled features.

In [13]:
# Logistic Regression
# your code here

*Observations:*
<!-- Fill in after running -->

## Section 7 — Model 2: Random Forest

Random Forest is an ensemble of decision trees trained via bagging. No scaling needed — splits are threshold-based on one feature at a time. OOB score gives a free out-of-sample estimate without touching the test set.

In [14]:
# Random Forest
# your code here

*Observations:*
<!-- Fill in after running -->

## Section 8 — Model 3: K-Nearest Neighbours

KNN makes no assumptions about decision boundary shape — it classifies by majority vote among the k nearest training points. Distance-based, so scaling is mandatory. We sweep k to find the best value before fitting the final model.

In [15]:
# KNN — k sweep
# your code here

In [16]:
# KNN — fit best k
# your code here

*Observations:*
<!-- Fill in after running -->

## Section 9 — Model Comparison

All three ROC curves on one axes, plus a summary table. The goal is not just to find the best number — it is to explain what each metric means in this context.

**Framing the trade-off:** A false negative here means predicting a passenger *survived* when they actually *died*. A false positive means predicting *death* when they survived. For a historical analysis, neither error carries real stakes — but in a real-world analogy (e.g. identifying high-risk patients) false negatives are typically more costly. Recall is the metric sensitive to false negatives.

In [17]:
# ROC curve overlay
# your code here

In [18]:
# Model comparison table
# your code here

*Interpretation:*
<!-- Fill in after running: which model wins on which metric, and what the trade-off means -->

## Section 10 — Conclusions

<!-- Fill in at end of session -->

**What worked:**

**What didn't / limitations:**

**What I'd try next:**
- Feature engineering: family size (SibSp + Parch + 1), title extraction from Name, deck from Cabin prefix
- sklearn Pipeline wrapping the preprocessing + model (Week 7)
- Cross-validation instead of a single train/test split (Week 6)
- Hyperparameter tuning with GridSearchCV (Week 9)
- XGBoost comparison (Week 13)